<a href="https://colab.research.google.com/github/kanickz/pre-built-models/blob/main/pre_build_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile

# 1. Download the file using Colab's native command (ignores security certificate bugs)
!wget --no-check-certificate https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip -O /content/dataset.zip

# 2. Unzip the file safely
with zipfile.ZipFile("/content/dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

# 3. Define your paths exactly where they were extracted
train_dir = "/content/cats_and_dogs_filtered/train"
val_dir = "/content/cats_and_dogs_filtered/validation"

print("Data is ready without any errors!")

--2026-06-12 16:53:28--  https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 209.85.145.207, 142.250.125.207, 209.85.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|209.85.145.207|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-06-12 16:53:28 ERROR 403: Forbidden.



BadZipFile: File is not a zip file

In [ ]:
import tensorflow_datasets as tfds
import os

print("Downloading and preparing dataset using tensorflow_datasets...")

# The 'cats_vs_dogs' dataset in TFDS might be slightly different in structure
# but contains the same images and is designed for this type of task.
(ds_train, ds_test), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'], # Splitting the 'train' data into train and validation
    shuffle_files=True,
    with_info=True,
    as_supervised=True,
)

# Helper function to process images
def preprocess_image(image, label):
    image = tf.image.resize(image, (150, 150))
    image = tf.cast(image, tf.float32) / 255.0 # Normalize to [0,1]
    return image, label

# Apply preprocessing to the datasets
BATCH_SIZE = 32

ds_train = ds_train.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Dataset prepared using tensorflow_datasets.")
print(f"Number of training samples: {ds_info.splits['train'].num_examples * 0.8}")
print(f"Number of validation samples: {ds_info.splits['train'].num_examples * 0.2}")

# Assign train_dir and val_dir for consistency with subsequent cells if they expect these
# Note: With TFDS, you directly use ds_train and ds_test. These directory variables
# might not be strictly necessary if subsequent cells are adapted.
# For now, we'll assign dummy paths or adapt the next cells.
# Given the original problem context, `ImageDataGenerator.flow_from_directory` is used.
# We will need to adapt the next cell (GD-fMxRptOv3) to use tf.data.Dataset directly.

# Placeholder for train_dir and val_dir for continuity, though not used directly for TFDS data pipeline
train_dir = "tensorflow_datasets_train_placeholder"
val_dir = "tensorflow_datasets_validation_placeholder"


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.LTIXAS_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Dataset prepared using tensorflow_datasets.
Number of training samples: 18609.600000000002
Number of validation samples: 4652.400000000001


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

# Setup training data generator with basic augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True
)

# Setup validation data generator (no augmentation)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Load images from the folders
train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras import layers, models

# 1. Load VGG16 without its top classification layer
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

# 2. Freeze VGG16 layers so they don't train
vgg_base.trainable = False

# 3. Add custom layers on top
model = models.Sequential([
    vgg_base,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax') # 2 classes: Cat or Dog
])

model.summary()

In [ ]:
from tensorflow.keras import optimizers

# Compile the model
model.compile(
    optimizer=optimizers.SGD(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(
    train_generator,
    epochs=3,
    validation_data=val_generator
)

In [ ]:
import numpy as np
import urllib.request
from tensorflow.keras.preprocessing import image

def predict(image_url):
    # Download the image
    urllib.request.urlretrieve(image_url, "/content/test.jpg")

    # Process the image to match the model size (150x150)
    img = image.load_img("/content/test.jpg", target_size=(150, 150))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # Get prediction
    preds = model.predict(x)

    # Display result
    if np.argmax(preds[0]) == 0:
        print("Prediction: CAT")
    else:
        print("Prediction: DOG")

# Test it with a sample dog image URL
test_url = "https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_1280.jpg"
predict(test_url)